# EX5 Bending Inverse Identification with LSTM-based RT-RPINN

- This notebook performs inverse identification for EX5 using the LSTM-based RT-RPINN field backbone.
- It reuses the same FE loader, material model, reduced-time recursion, and inverse losses as the RT-RPINN inverse script.

Run the cells in order. The paths are kept consistent with the original EX5 Python scripts.


## Script Notes

Example 5 (INVERSE, LSTM) — identify the time-temperature shift parameters
(WLF C1, C2; Arrhenius Ea/R) of the bending shape-memory cycle with an LSTM-PINN.

LSTM counterpart of inverse_spatiotemporal_pinn_ex5.py. It shares ALL the physics
(FEDataLoader, InverseShiftParams, the q-recursion, compute_stress, the free-recovery
master-curve loss, the crossover-continuity penalty, and the trainable Prony spectrum)
with the MLP/spatiotemporal inverse solver — only the displacement FIELD backbone
changes (LSTM over the time sequence instead of a coordinate MLP).

WHY THE LSTM VERSION IS SIMPLE HERE
-----------------------------------
In EX5 the shift-parameter identification is FULLY DECOUPLED from the field network:
  * the PRIMARY signal is the free-recovery UR curve, matched through the precomputed
    creep-recovery master curve R(ξ) — it depends on a_T(C1,C2,Ea/R) and the trainable
    Prony spectrum, not on the network;
  * the weak stress / stress-increment anchors are computed from the FE-MEASURED strain
    (Abaqus LE → q(a_T) → σ), again with no network gradient.
So the LSTM only has to RECONSTRUCT the displacement field (data + clamped-left BC).
There is no need for per-step displacement-gradient strain (that small-strain measure is
invalid under the ~270° rotation anyway — lambda_strain=0), so the whole sequence can be
run in one LSTM pass (no O(N_t²) BPTT issue).

NO REACTION MOMENT (RM): lambda_rf=0, lambda_obs=0 — identical no-RM config to the MLP
inverse. C2 is held fixed at its known value (structural identifiability: the WLF C1-C2
valley is non-identifiable together); fix_c2=True. The shift parameters and
{g2,g3,g4,g_inf} are identified jointly.

Outputs: ex5_lstm_inverse_{loss,param}_history.csv, ex5_lstm_inverse_model.pth,
         ex5_lstm_inverse_training_history.png, ex5_lstm_inverse_training_log.txt


In [ ]:
# Notebook path setup
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'EX-5-RESULTS').exists() and (NOTEBOOK_DIR / 'EX5' / 'EX-5-RESULTS').exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / 'EX5'
NOTEBOOK_DIR = NOTEBOOK_DIR.resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))
print(f'EX5 notebook directory: {NOTEBOOK_DIR}')


## Imports and definitions


In [ ]:
import sys
import math
import time
import argparse
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

from inverse_spatiotemporal_pinn_ex5 import (
    _Tee,
    device,
    set_global_seed,
    FEDataLoader,
    InverseShiftParams,
    InversePINNSolver,
    plot_training_history,
)


def _parse_dims(s):
    """Parse a comma-separated list of ints, e.g. '128,128'."""
    return tuple(int(v) for v in str(s).split(',') if v.strip())


## LSTM displacement-field backbone (whole-sequence forward)


In [ ]:
# ---------------------------------------------------------------------------
class LSTMInverseEX5(nn.Module):
    """
    Per spatial node: encode (x,y,z, 45° fiber-local, spatial Fourier) → features,
    broadcast over the time sequence, append time-Fourier + temperature, run an LSTM,
    decode to u(t). Output scaled by `output_scale` (≈ 1.1·max|u|) to predict an O(1)
    field — the steep clamped→free gradient over the 0→108 mm, 270° curl is otherwise
    hard to form (spectral bias). Mirrors the forward LSTM backbone.
    """

    def __init__(self, spatial_hidden=(128, 128), lstm_hidden=128, lstm_layers=2,
                 n_fourier_space=5, n_fourier_time=5, output_scale=1.0, dropout=0.0):
        super().__init__()
        c = math.cos(math.radians(45.0)); s = math.sin(math.radians(45.0))
        self.register_buffer("cos_theta", torch.tensor(c, dtype=torch.float32))
        self.register_buffer("sin_theta", torch.tensor(s, dtype=torch.float32))
        self.register_buffer("sfreqs",
            2.0 * math.pi * torch.arange(1, n_fourier_space + 1, dtype=torch.float32))
        self.register_buffer("tfreqs",
            2.0 * math.pi * torch.arange(1, n_fourier_time + 1, dtype=torch.float32))
        self.register_buffer("output_scale",
            torch.tensor(float(output_scale), dtype=torch.float32))

        self.lstm_hidden = lstm_hidden
        self.lstm_layers = lstm_layers

        spatial_in = 5 + 3 * 2 * n_fourier_space
        dims = [spatial_in, *spatial_hidden]
        enc = []
        for i in range(len(dims) - 1):
            enc.append(nn.Linear(dims[i], dims[i + 1]))
            enc.append(nn.Tanh())
            if dropout > 0:
                enc.append(nn.Dropout(dropout))
        self.spatial_encoder = nn.Sequential(*enc)

        lstm_in = dims[-1] + (2 * n_fourier_time + 1) + 1
        self.lstm = nn.LSTM(input_size=lstm_in, hidden_size=lstm_hidden,
                            num_layers=lstm_layers, batch_first=True,
                            dropout=dropout if lstm_layers > 1 else 0.0)
        self.decoder = nn.Sequential(
            nn.Linear(lstm_hidden, lstm_hidden), nn.Tanh(),
            nn.Linear(lstm_hidden, 3),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def _spatial_feats(self, x, y, z):
        """x,y,z: (B,1) normalised coords → (B, F) spatial features."""
        x_local = x * self.cos_theta + y * self.sin_theta
        y_local = -x * self.sin_theta + y * self.cos_theta
        feats = [x, y, z, x_local, y_local]
        for v in (x, y, z):
            for f in self.sfreqs:
                feats.append(torch.sin(v * f)); feats.append(torch.cos(v * f))
        return self.spatial_encoder(torch.cat(feats, dim=1))

    def forward(self, x, y, z, t_seq, T_seq):
        """
        Args:
            x, y, z:      (B,1) normalised node coords (constant across the sequence)
            t_seq, T_seq: (S,)  normalised time / temperature for the sequence
        Returns:
            u: (B, S, 3) displacement in mm
        """
        B = x.shape[0]; S = t_seq.shape[0]
        sf = self._spatial_feats(x, y, z).unsqueeze(1).expand(B, S, -1)  # (B,S,F)
        t = t_seq.view(1, S, 1).expand(B, S, 1)
        t_enc = torch.cat(
            [torch.sin(t * f) for f in self.tfreqs] +
            [torch.cos(t * f) for f in self.tfreqs] + [t], dim=-1)
        T = T_seq.view(1, S, 1).expand(B, S, 1)
        inp = torch.cat([sf, t_enc, T], dim=-1)
        out, _ = self.lstm(inp)
        return self.decoder(out) * self.output_scale


## LSTM inverse solver — inherits all physics from InversePINNSolver


In [ ]:
# ---------------------------------------------------------------------------
class InverseLSTMSolverEX5(InversePINNSolver):
    """
    LSTM-PINN solver for EX5 inverse shift + Prony parameter identification (no RM).

    Overrides train(): the LSTM reconstructs the displacement field, while the
    shift parameters are identified by the (network-independent) free-recovery
    master-curve loss + the FE-strain stress/increment anchors + the crossover
    continuity penalty — all inherited from InversePINNSolver.
    """

    def train(self, epochs=10000, batch_size=512,
              lr_network=1e-3, lr_params=1e-2,
              log_interval=100, seq_length=60, adaptive_lr=True):

        start_time = time.time()
        full_data = self.fe_loader.full_data


## Displacement reference scale


In [ ]:
x_max = self.bounds['x_max']
        right_data = full_data[np.abs(full_data['X'].values - x_max) < 1e-4]
        u_mag = np.linalg.norm(right_data[['U1', 'U2', 'U3']].values, axis=1)
        u_ref = float(np.max(u_mag))
        self.u_ref_sq = (u_ref + 1e-10) ** 2
        print(f"\nU scale (max|u|): {u_ref:.4f} mm")
        print("RF supervision: disabled (lambda_rf=0)" if self.lambda_rf <= 0.0
              else f"RF lambda={self.lambda_rf}")


## PRIMARY a_T signal: free-recovery (step 4) UR-recovery master curve.


In [ ]:
if self.lambda_recovery > 0.0:
            self._prepare_recovery_target()


## Stable right-face node set (keeps the sampled node set consistent).


In [ ]:
right_nodes = full_data[np.abs(full_data['X'].values - x_max) < 1e-4]
        n_rf_points = 512
        for col in ['NodeLabel', 'Node', 'NID']:
            if col in right_nodes.columns:
                n_rf_points = max(1, int(right_nodes[col].nunique()))
                break
        self.initialize_fixed_rf_points(n_rf_points=n_rf_points)

        for p in self.model.parameters():
            p.requires_grad_(True)
        for p in self.mat.parameters():
            p.requires_grad_(True)


## Structural fix: C2 anchored at its known value, frozen the whole run.


In [ ]:
if getattr(self, 'fix_c2', False) and hasattr(self.mat, 'log_C2'):
            self.mat.log_C2.requires_grad_(False)
            print(f"  [fix_c2] C2 frozen at {self.mat.C2.item():.2f} K (true {self.mat.TRUE_C2})")

        shift_params = list(self.mat.parameters())   # log_C1, log_C2, log_Ea_R, logits_g_free

        optimizer = Adam([
            {'params': self.model.parameters(), 'lr': lr_network},
            {'params': shift_params,            'lr': lr_params},
        ])
        scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)
        shift_lr_floor = max(6e-5, 0.25 * lr_params)
        net_lr_floor = max(1e-5, 0.05 * lr_network)


## Plateau-triggered base-LR reduction (tracks the recovery a_T signal).


In [ ]:
best_id_monitor = float('inf')
        plateau_count = 0
        plateau_patience = max(120, epochs // 25)
        id_improve_tol = 1e-3
        lr_reduce_net = 0.85
        lr_reduce_shift = 0.90

        print(f"\nStarting EX5 LSTM inverse training ({epochs} epochs)...")
        print(f"  seq_length={seq_length}, batch_size={batch_size}")
        print(f"  Initial C1={self.mat.C1.item():.2f} (true {self.mat.TRUE_C1})")
        print(f"  Initial C2={self.mat.C2.item():.2f} (true {self.mat.TRUE_C2})")
        print(f"  Initial Ea_R={self.mat.Ea_R.item():.1f} (true {self.mat.TRUE_Ea_R})")
        print("-" * 80)

        def _clamp_shift_params():
            if hasattr(self.mat, 'log_C1'):
                self.mat.log_C1.data.clamp_(math.log(4.0), math.log(30.0))
            if hasattr(self.mat, 'log_C2'):
                self.mat.log_C2.data.clamp_(math.log(10.0), math.log(100.0))
            if hasattr(self.mat, 'log_Ea_R'):
                self.mat.log_Ea_R.data.clamp_(math.log(3000.0), math.log(50000.0))

        for epoch in range(epochs):
            self.model.train()
            optimizer.zero_grad()

            seq_data = self.sample_temporal_sequence(
                full_data, batch_size=batch_size, seq_length=seq_length,
                required_node_ids=self.rf_node_ids, node_id_col=self.rf_node_col,
            )
            coords_seq      = seq_data['coords_seq']
            u_data_seq      = seq_data['u_data_seq']
            strain_data_seq = seq_data['strain_data_seq']
            stress_data_seq = seq_data['stress_data_seq']
            sampled_times   = seq_data['times']
            T_seq           = seq_data['T_seq']
            dt_seq          = seq_data['dt_seq']
            N_pts = seq_data['N_pts']
            N_t   = seq_data['N_t']
            left_indices = seq_data['left_indices']

            # ---- Normalised, time-constant node coords (X,Y,Z are reference coords) ----
            xyz = coords_seq[0][:, 0:3]
            xh = ((xyz[:, 0:1] - self.x_min) / self.Lx)
            yh = ((xyz[:, 1:2] - self.y_min) / self.Ly)
            zh = ((xyz[:, 2:3] - self.z_min) / self.Lz)

            t_arr = torch.tensor(sampled_times, dtype=torch.float32, device=device)
            T_arr = torch.tensor(T_seq, dtype=torch.float32, device=device)
            th_seq = (t_arr - self.t_min) / self.T_max          # (N_t,)
            Th_seq = (T_arr - self.Temp_min) / self.Temp_range   # (N_t,)

            # ---- LSTM forward over the whole sequence (field reconstruction) ----
            u_pred = self.model(xh, yh, zh, th_seq, Th_seq)      # (N_pts, N_t, 3)
            u_data_stack = torch.stack(u_data_seq, dim=0)        # (N_t, N_pts, 3)
            u_pred_perm = u_pred.permute(1, 0, 2)                # (N_t, N_pts, 3)

            loss_data = torch.mean((u_pred_perm - u_data_stack) ** 2) / self.u_ref_sq
            if len(left_indices) > 0:
                li = torch.tensor(left_indices, dtype=torch.long, device=device)
                loss_bc_left = torch.mean(u_pred[li, :, :] ** 2) / self.u_ref_sq
            else:
                loss_bc_left = torch.zeros((), device=device)

            # ---- Shift-parameter ID (network-independent) ------------------------


## (1) Weak stress / stress-increment anchors from the FE-measured strain.


In [ ]:
loss_stress = torch.zeros((), device=device)
            loss_stress_rate = torch.zeros((), device=device)
            n_stress_rate = 0
            use_fe_stress = (
                self.identify_from_fe_strain and self.lambda_stress > 0.0
                and all(s is not None for s in strain_data_seq)
                and all(s is not None for s in stress_data_seq)
            )
            if use_fe_stress:
                strain_fe_mat_seq = [strain_data_seq[n] for n in range(N_t)]
                strain_fe_mat_tensor = torch.stack(strain_fe_mat_seq, dim=0)
                T_for_q = T_arr.unsqueeze(1).expand(N_t, N_pts)   # (N_t, N_pts)
                q_fe_seq = self.compute_q_recursive(strain_fe_mat_tensor, dt_seq,
                                                    T_for_q, self.mat)
                sigma_ref_sq = 50.0 ** 2
                sigma_rate_ref_sq = 10.0 ** 2
                prev_stress_pred = None
                prev_stress_fe = None
                for n in range(N_t):
                    t_val = sampled_times[n]


## Material-frame stress (FE S is in the fiber frame): to_global=False.


In [ ]:
stress_pred = self.compute_stress(strain_fe_mat_seq[n],
                                                      q_fe_seq[n], to_global=False)


## a_T-sensitivity window weighting (crossover bands carry the signal).


In [ ]:
w_aT = 1.0
                    if 45.0 <= t_val < 65.0:       # cooling crossover (T≈317K @48.4s)
                        w_aT = 2.2
                    elif 20.0 <= t_val < 45.0:      # early cooling (WLF)
                        w_aT = 1.3
                    elif 65.0 <= t_val < 70.0:      # late cooling (near-frozen)
                        w_aT = 1.2
                    elif 85.0 <= t_val <= 100.0:    # recovery Tg re-crossing (~88.6s)
                        w_aT = 2.0
                    loss_stress = loss_stress + w_aT * torch.mean(
                        (stress_pred - stress_data_seq[n]) ** 2) / sigma_ref_sq


## model-mismatch offset → the clean a_T signal.


In [ ]:
if (self.lambda_stress_rate > 0.0 and prev_stress_pred is not None):
                        dsig_pred = stress_pred - prev_stress_pred
                        dsig_fe = stress_data_seq[n] - prev_stress_fe
                        loss_stress_rate = loss_stress_rate + w_aT * torch.mean(
                            (dsig_pred - dsig_fe) ** 2) / sigma_rate_ref_sq
                        n_stress_rate += 1
                    prev_stress_pred = stress_pred
                    prev_stress_fe = stress_data_seq[n]
                loss_stress = loss_stress / N_t
                loss_stress_rate = loss_stress_rate / max(1, n_stress_rate)


## (2) PRIMARY a_T signal: free-recovery UR master-curve loss.


In [ ]:
loss_recovery = (self.compute_recovery_loss()
                             if self.lambda_recovery > 0.0
                             else torch.zeros((), device=device))


## (3) Crossover-continuity + range penalty (links C1, C2, Ea/R).


In [ ]:
loss_penalty = self.mat.compute_parameter_penalty()

            loss_total = (
                self.lambda_data        * loss_data +
                self.lambda_bc_left     * loss_bc_left +
                self.lambda_stress      * loss_stress +
                self.lambda_stress_rate * loss_stress_rate +
                self.lambda_recovery    * loss_recovery +
                50.0                    * loss_penalty
            )

            if not torch.isfinite(loss_total):
                print(f"[Epoch {epoch:5d}] Non-finite loss; C1={self.mat.C1.item()} "
                      f"Ea_R={self.mat.Ea_R.item()}")
                break

            loss_total.backward()


## Structural fix: kill any C2 gradient every step (entire run).


In [ ]:
if getattr(self, 'fix_c2', False) and hasattr(self.mat, 'log_C2') \
                    and self.mat.log_C2.grad is not None:
                self.mat.log_C2.grad.zero_()

            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            torch.nn.utils.clip_grad_norm_(shift_params, max_norm=0.8)

            optimizer.step()
            scheduler.step()


## LR floors (keep shift LR from collapsing into a wrong local min).


In [ ]:
if optimizer.param_groups[1]['lr'] < shift_lr_floor:
                optimizer.param_groups[1]['lr'] = shift_lr_floor
            if optimizer.param_groups[0]['lr'] < net_lr_floor:
                optimizer.param_groups[0]['lr'] = net_lr_floor

            if adaptive_lr:
                id_monitor = 0.0
                if self.lambda_recovery > 0.0:
                    id_monitor += 10.0 * loss_recovery.detach().item()
                if self.lambda_stress_rate > 0.0:
                    id_monitor += 4.0 * loss_stress_rate.detach().item()
                if self.lambda_stress > 0.0:
                    id_monitor += 0.3 * loss_stress.detach().item()
                if id_monitor < best_id_monitor * (1.0 - id_improve_tol):
                    best_id_monitor = id_monitor
                    plateau_count = 0
                else:
                    plateau_count += 1
                    if plateau_count >= plateau_patience:
                        scheduler.base_lrs[0] = max(net_lr_floor, scheduler.base_lrs[0] * lr_reduce_net)
                        scheduler.base_lrs[1] = max(shift_lr_floor, scheduler.base_lrs[1] * lr_reduce_shift)
                        optimizer.param_groups[0]['lr'] = max(net_lr_floor, optimizer.param_groups[0]['lr'] * lr_reduce_net)
                        optimizer.param_groups[1]['lr'] = max(shift_lr_floor, optimizer.param_groups[1]['lr'] * lr_reduce_shift)
                        plateau_count = 0
                        print(f"[Epoch {epoch:5d}] adaptive-lr reduce: "
                              f"net={optimizer.param_groups[0]['lr']:.2e} "
                              f"shift={optimizer.param_groups[1]['lr']:.2e} "
                              f"id_monitor={id_monitor:.4e}")
            _clamp_shift_params()

            # ---- History ----
            self.loss_history.append(loss_total.item())
            self.loss_components_history.append({
                'data':     loss_data.item(),
                'recovery': loss_recovery.item(),
                'stress':   loss_stress.item(),
                'stress_rate': loss_stress_rate.item(),
                'bc_left':  loss_bc_left.item(),
                'penalty':  loss_penalty.item(),
            })
            self.param_history.append({
                'C1':   self.mat.C1.item(),
                'C2':   self.mat.C2.item(),
                'Ea_R': self.mat.Ea_R.item(),
                'g2':    self.mat.g_matrix[2].item(),
                'g3':    self.mat.g_matrix[3].item(),
                'g4':    self.mat.g_matrix[4].item(),
                'g_inf': self.mat.g_matrix[6].item(),
            })

            if epoch % log_interval == 0 or epoch == epochs - 1:
                lr_net = optimizer.param_groups[0]['lr']
                lr_sft = optimizer.param_groups[1]['lr']
                _terms = [f"data={loss_data.item():.4e}"]
                if self.lambda_recovery > 0.0:
                    _terms.append(f"recov={loss_recovery.item():.4e}")
                if self.lambda_stress > 0.0:
                    _terms.append(f"stress={loss_stress.item():.4e}")
                if self.lambda_stress_rate > 0.0:
                    _terms.append(f"srate={loss_stress_rate.item():.4e}")
                print(f"[Epoch {epoch:5d}] total={loss_total.item():.4e} | "
                      + " ".join(_terms))
                print(f"  [Shift] C1={self.mat.C1.item():.3f}(T:{self.mat.TRUE_C1})  "
                      f"C2={self.mat.C2.item():.2f}(T:{self.mat.TRUE_C2})  "
                      f"Ea_R={self.mat.Ea_R.item():.0f}(T:{self.mat.TRUE_Ea_R})")
                _gm = self.mat.g_matrix.detach().cpu().tolist()
                _tg = self.mat.TRUE_G
                print(
                    f"  [Prony] g2={_gm[2]:.3f}(T:{_tg[2]})  g3={_gm[3]:.3f}(T:{_tg[3]})  "
                    f"g4={_gm[4]:.3f}(T:{_tg[4]})  g_inf={_gm[6]:.4f}(T:{_tg[6]})"
                )
                print(f"  [LR] net={lr_net:.2e} shift={lr_sft:.2e}  "
                      f"t=[{sampled_times[0]:.1f},{sampled_times[-1]:.1f}]s N_t={N_t}")

        print(f"\nLSTM inverse training complete: {time.time() - start_time:.1f}s")


## Run configuration


In [ ]:
# ---------------------------------------------------------------------------
def main():
    parser = argparse.ArgumentParser(
        description="EX5 bending inverse shift + Prony parameter identification, LSTM-PINN (no RM).")
    parser.add_argument('--epochs', type=int, default=10000)
    parser.add_argument('--batch-size', type=int, default=512,
                        help='Spatial node batch per sampled sequence.')
    parser.add_argument('--seq-length', type=int, default=60,
                        help='Temporal sequence length sampled each epoch.')
    parser.add_argument('--lr-network', type=float, default=1e-3)
    parser.add_argument('--lr-params', type=float, default=2e-3,
                        help='Shift-parameter LR. Lowered 1e-2→2e-3 for a GRADUAL '
                             'recovery-driven C1/Ea_R convergence (visible trajectory, '
                             'not a ~100-epoch snap). Use 1e-3 for even more gradual.')
    parser.add_argument('--spatial-hidden', type=_parse_dims, default=(128, 128))
    parser.add_argument('--lstm-hidden', type=int, default=128)
    parser.add_argument('--lstm-layers', type=int, default=2)
    parser.add_argument('--n-spatial', type=int, default=3000,
                        help='Number of spatial nodes kept (0/neg → all).')
    parser.add_argument('--n-temporal', type=int, default=150,
                        help='Number of temporal frames kept (0/neg → use --stride).')
    parser.add_argument('--stride', type=int, default=5)
    parser.add_argument('--no-adaptive-lr', dest='adaptive_lr', action='store_false')
    parser.add_argument('--seed', type=int, default=42)
    args = parser.parse_args(args=[])
    set_global_seed(args.seed)
    n_spatial = args.n_spatial if args.n_spatial and args.n_spatial > 0 else None
    n_temporal = args.n_temporal if args.n_temporal and args.n_temporal > 0 else None

    print("=" * 70)
    print("EX5 FORWARD + INVERSE PARAMETER IDENTIFICATION — LSTM-PINN")
    print("Bending shape-memory cycle (95x13x2 beam, end rotation UR=4.71 rad)")
    print("Configuration: NO reaction-moment (RM) supervision  [先不带rm约束]")
    print("=" * 70)
    print(f"Seed: {args.seed}")

    script_dir = NOTEBOOK_DIR
    data_dir   = script_dir / 'EX-5-RESULTS'
    rf_file    = script_dir / 'ex-5-Bending-RM.csv'
    u_file     = script_dir / 'ex-5-Bending-UR.csv'
    ft_file    = script_dir / 'frame-time.csv'

    print("\nLoading FE data...")
    fe_loader = FEDataLoader(
        data_dir, rf_file, u_file, ft_file, stride=args.stride,
        n_spatial=n_spatial, spatial_seed=args.seed, n_temporal=n_temporal)
    fe_loader.load_all_data()
    bounds = fe_loader.get_domain_bounds()

    print("\nDomain bounds:")
    for k, v in bounds.items():
        print(f"  {k}: {v:.2f}")


## Output displacement scale for the O(1) field prediction.


In [ ]:
u_mag = np.linalg.norm(
        fe_loader.full_data[['U1', 'U2', 'U3']].values.astype(float), axis=1)
    u_scale = float(1.1 * np.max(u_mag))
    print(f"\nOutput displacement scale: {u_scale:.2f} mm (1.1 x max|u|)")

    print("\nInitialising identified parameters:")
    print(f"  C1   : init=8.0     true={InverseShiftParams.TRUE_C1}   (identified)")
    print(f"  C2   : FIXED=45.6   true={InverseShiftParams.TRUE_C2}   (held: non-identifiable)")
    print(f"  Ea/R : init=20000   true={InverseShiftParams.TRUE_Ea_R} (identified, via continuity)")
    print(f"  Prony: g2,g3,g4,g_inf identified jointly (init≈0.350/0.250/0.080/0.020,")
    print(f"         true 0.306/0.358/0.034/0.002); g0,g1,g5 held fixed")

    mat_params = InverseShiftParams().to(device)

    model = LSTMInverseEX5(
        spatial_hidden=args.spatial_hidden,
        lstm_hidden=args.lstm_hidden,
        lstm_layers=args.lstm_layers,
        output_scale=u_scale,
    ).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"\nLSTM-PINN parameters: {n_params:,}")
    print(f"  spatial hidden={args.spatial_hidden}  "
          f"LSTM={args.lstm_layers}x{args.lstm_hidden}")

    solver = InverseLSTMSolverEX5(
        model, mat_params, fe_loader, bounds,
        lambda_data=8.0,     # network displacement fit (field reconstruction)
        lambda_strain=0.0,   # network-differentiated strain invalid under rotation
        lambda_recovery=80.0,  # PRIMARY signal for BOTH a_T AND the Prony spectrum
        lambda_stress=1.0,   # weak anchor only (σ~0.4 MPa floor; spectrum now from recovery)
        lambda_stress_rate=2.0,  # weak: relaxation increment
        lambda_pde=0.0,
        lambda_bc_left=1.0,  # clamp u=0 (rotation-agnostic, valid)
        lambda_bc_right=0.0,
        lambda_traction=0.0,
        lambda_rf=0.0,       # NO reaction-moment supervision (EX5 无RM约束)
        lambda_obs=0.0,
        lambda_obs_rate=0.0,
        finite_strain=False,
        identify_from_fe_strain=True,   # stress from measured strain + a_T(C1,C2,Ea/R)
        fix_c2=True,         # WLF (C1,C2) non-identifiable together → hold C2 at known 45.6
    )

    solver.train(
        epochs=args.epochs,
        batch_size=args.batch_size,
        lr_network=args.lr_network,
        lr_params=args.lr_params,
        log_interval=100,
        seq_length=args.seq_length,
        adaptive_lr=args.adaptive_lr,
    )


## Save


In [ ]:
print("\nSaving results...")
    solver.save_loss_history(script_dir / 'ex5_lstm_inverse_loss_history.csv')
    solver.save_param_history(script_dir / 'ex5_lstm_inverse_param_history.csv')
    plot_training_history(solver, script_dir,
                          filename='ex5_lstm_inverse_training_history.png')

    model_path = script_dir / 'ex5_lstm_inverse_model.pth'
    torch.save({
        'model_state_dict':      model.state_dict(),
        'mat_params_state_dict': mat_params.state_dict(),
        'final_params': {
            'C1':   mat_params.C1.item(),
            'C2':   mat_params.C2.item(),
            'Ea_R': mat_params.Ea_R.item(),
            'g2':    mat_params.g_matrix[2].item(),
            'g3':    mat_params.g_matrix[3].item(),
            'g4':    mat_params.g_matrix[4].item(),
            'g_inf': mat_params.g_matrix[6].item(),
        },
        'true_params': {
            'C1':   InverseShiftParams.TRUE_C1,
            'C2':   InverseShiftParams.TRUE_C2,
            'Ea_R': InverseShiftParams.TRUE_Ea_R,
            'g2':    InverseShiftParams.TRUE_G[2],
            'g3':    InverseShiftParams.TRUE_G[3],
            'g4':    InverseShiftParams.TRUE_G[4],
            'g_inf': InverseShiftParams.TRUE_G[6],
        }
    }, model_path)
    print(f"Model saved to {model_path}")

    print("\n" + "=" * 70)
    print("FINAL IDENTIFIED PARAMETERS (EX5 LSTM, no RM — shift + Prony spectrum)")
    print("=" * 70)
    print(f"  C1   = {mat_params.C1.item():.4f}   (true: {InverseShiftParams.TRUE_C1})")
    print(f"  C2   = {mat_params.C2.item():.4f} K (true: {InverseShiftParams.TRUE_C2} K)")
    print(f"  Ea/R = {mat_params.Ea_R.item():.2f} K (true: {InverseShiftParams.TRUE_Ea_R} K)")
    c1_err  = abs(mat_params.C1.item()  - InverseShiftParams.TRUE_C1)  / InverseShiftParams.TRUE_C1 * 100
    c2_err  = abs(mat_params.C2.item()  - InverseShiftParams.TRUE_C2)  / InverseShiftParams.TRUE_C2 * 100
    ear_err = abs(mat_params.Ea_R.item()- InverseShiftParams.TRUE_Ea_R)/ InverseShiftParams.TRUE_Ea_R * 100
    print(f"\n  Relative errors: C1={c1_err:.1f}%  C2={c2_err:.1f}%  Ea/R={ear_err:.1f}%")

    _gm = mat_params.g_matrix.detach().cpu().tolist()
    _tg = InverseShiftParams.TRUE_G
    print("\n  Prony spectrum (identified: g2,g3,g4,g_inf; fixed: g0,g1,g5):")
    for _k, _name in ((2, 'g2'), (3, 'g3'), (4, 'g4'), (6, 'g_inf')):
        _e = abs(_gm[_k] - _tg[_k]) / max(_tg[_k], 1e-9) * 100
        print(f"    {_name:5s} = {_gm[_k]:.4f}   (true: {_tg[_k]:.4f})   err={_e:.1f}%")


if __name__ == "__main__":
    _script_dir = NOTEBOOK_DIR
    _log_path = _script_dir / 'ex5_lstm_inverse_training_log.txt'
    _tee = _Tee(sys.stdout, _log_path)
    sys.stdout = _tee
    try:
        main()
    finally:
        sys.stdout = _tee._orig
        _tee.close()
        _tee._orig.write(f"\nLog saved to: {_log_path}\n")
